In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import HuberRegressor
from tqdm import tqdm
import statsmodels.api as sm
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
with open("data/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
data.head()

In [ ]:
def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y-y_pred)**2) / np.sum(y**2)
    return r2

In [ ]:
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

class CNN(nn.Module):
    def __init__(self, input_channels, output_size):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 23, 64)  # 32 channels and sequence length after pooling
        self.fc2 = nn.Linear(64, output_size)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

pred_y_list = []
true_y_list = []

for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    
    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')
    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')


    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    if train_data.empty or test_data.empty:
        print(f"No data available for the range {start_date_train.strftime('%Y-%m')} to {end_date_train.strftime('%Y-%m')} or {start_date_test.strftime('%Y-%m')} to {end_date_test.strftime('%Y-%m')}")
        continue
    

    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
   
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)  
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)
    
  
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=200, shuffle=True)

   
    input_channels = 1
    output_size = 1
    learning_rate = 0.0001
    n_epochs = 100
    patience = 10

    model = CNN(input_channels, output_size)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    best_loss = float('inf')
    best_model_state = None
    counter = 0

    for epoch in range(n_epochs):
        model.train()
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_test_tensor)
            val_loss = criterion(val_outputs, y_test_tensor).item()
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = model.state_dict().copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break
        
        if epoch % 10 == 0:
            train_outputs = model(X_train_tensor)
            train_loss = criterion(train_outputs, y_train_tensor).item()
            print(f"Epoch {epoch}, train loss: {train_loss}, val loss: {val_loss}")

   
    model.load_state_dict(best_model_state)

    model.eval()
    with torch.no_grad():
        y_pred = model(X_test_tensor)
        test_loss = criterion(y_pred, y_test_tensor).item()

  
    r2 = r2_score(y_test_tensor.numpy(), y_pred.numpy())
    print(f'Year {year}: R2 Score: {r2}')
    
    pred_y_list.extend(y_pred.numpy())
    true_y_list.extend(y_test_tensor.numpy())

overall_r2 = r2_score(true_y_list, pred_y_list)
print(f'Overall R2 Score: {overall_r2}')